# LoCoMo Eval for Fastrr — Step by Step

Run each cell in order. Uses the same config as Fastrr (Ollama by default).

1. **Setup** — Imports, paths, config
2. **Load dataset** — Load locomo10.json
3. **Create storage** — Repo + Fastrr memory
4. **Step 1: Ingest** — Load conversations into memory
5. **Inspect** — See what was stored
6. **Build QA agents** — Answer generator + grader
7. **Step 2a: One question** — Recall → generate → grade for a single question
8. **Step 2b: All questions** — Run full QA loop
9. **Step 3: Results** — Aggregate and save

## 1. Setup

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from collections import defaultdict

# Paths: detect repo root and add to path so evals + fastrr import
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "locomo_eval.ipynb").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent  # cwd is evals/locomo
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

EVALS_DIR = PROJECT_ROOT / "evals"
DATASETS_DIR = EVALS_DIR / "datasets"
DATASET_PATH = DATASETS_DIR / "locomo10.json"
OUTPUT_DIR = EVALS_DIR / "output"

# Disable Agno telemetry before importing Agno
os.environ.setdefault("AGNO_TELEMETRY", "false")

from agno.agent import Agent
from fastrr import Fastrr
from fastrr.core.config import FastrrConfig
from evals.locomo.ingest import ingest_locomo
from evals.fake_repo import FakeRepoManager

cfg = FastrrConfig()
print(f"Config: provider={cfg.provider} model={cfg.model}")
print(f"Dataset: {DATASET_PATH}")
print(f"Exists: {DATASET_PATH.exists()}")

Config: provider=ollama model=qwen3:1.7b
Dataset: /Users/tarunkhilani/Documents/projects/fastrr/evals/datasets/locomo10.json
Exists: True


## 2. Load dataset

In [2]:
with open(DATASET_PATH) as f:
    data = json.load(f)

num_users = min(10, len(data))
total_qa = sum(
    len([q for q in data[i].get("qa", []) if q.get("category") != 5])
    for i in range(num_users)
)
print(f"Conversations: {num_users}")
print(f"Total QA (excl. abstention): {total_qa}")
print(f"Sample keys per conversation: {list(data[0].keys())}")

Conversations: 10
Total QA (excl. abstention): 1540
Sample keys per conversation: ['qa', 'conversation', 'event_summary', 'observation', 'session_summary', 'sample_id']


## 3. Create storage (FakeRepoManager + Fastrr)

In [3]:
root = OUTPUT_DIR / "storage"
storage_path = root / "repo"
worktree_root = root / "users"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
storage_path.mkdir(parents=True, exist_ok=True)
worktree_root.mkdir(parents=True, exist_ok=True)

repo = FakeRepoManager(root)
memory = Fastrr(storage_path=storage_path, worktree_root=worktree_root, repo_manager=repo)

print(f"Storage root: {root}")
print("FakeRepoManager (no Git)")

Storage root: /Users/tarunkhilani/Documents/projects/fastrr/evals/output/storage
FakeRepoManager (no Git)


## 4. Step 1: Ingest

In [4]:
def _format_message(msg: dict, session_date: str) -> str:
    """Format a single message for remember()."""
    speaker = msg.get("speaker", "Unknown")
    text = msg.get("text", "")
    blip = msg.get("blip_caption")
    img_desc = f" (image: {blip})" if blip else ""
    return f"[{session_date}] {speaker}: {text}{img_desc}"

path = Path(DATASET_PATH)
with open(path) as f:
    data = json.load(f)

user_ids: list[str] = []
for group_idx in range(min(num_users, len(data))):
    sample = data[group_idx]
    conversation = sample.get("conversation", {})
    user_id = f"locomo_{group_idx}"
    user_ids.append(user_id)

    for session_idx in range(35):
        session_key = f"session_{session_idx}"
        session = conversation.get(session_key)
        if session is None:
            continue

        date_key = f"session_{session_idx}_date_time"
        session_date = conversation.get(date_key, "unknown")

        for msg in session:
            content = _format_message(msg, session_date)
            print(content)
            memory.remember(user_id, content)
        break
    break
            

[1:56 pm on 8 May, 2023] Caroline: Hey Mel! Good to see you! How have you been?


DEBUG ****** Agent ID: c8165fb7-fd36-4717-8341-95a87729e2c7 ******

DEBUG Creating new AgentSession: 1d2ca7a5-67c7-4c14-802d-b29a7aa57a71

DEBUG ** Agent Run Start: adba65ba-bd4d-4664-a910-e2094fbd943b ***

DEBUG Processing tools for model

...

DEBUG ************************  METRICS  *************************

DEBUG ------------------- Ollama Response End --------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 3cc67682-35bb-4fed-8b04-cede26ffe5d4 ****

The memory for user 'locomo_0' has been successfully stored in `facts.md`. No further actions are required.


In [ ]:
start = time.monotonic()
ingest_locomo(memory, DATASET_PATH, num_users=num_users, log=print)
elapsed = time.monotonic() - start
print(f"\nDone. Ingested {num_users} conversations in {elapsed:.1f}s")

## 5. Inspect what was stored

In [ ]:
users = memory.list_users()
print(f"Users: {users}")

# Peek at first user's file
workspace = Path(repo.ensure_user_worktree("locomo_0"))
files = list(workspace.rglob("*"))
print(f"\nlocomo_0 files: {[p.name for p in files if p.is_file()]}")
if files:
    f = next(p for p in files if p.is_file())
    print(f"\nFirst 500 chars of {f.name}:")
    print(f.read_text()[:500])

## 6. Build QA agents (answer generator + grader)

In [ ]:
if cfg.provider == "openrouter":
    from agno.models.openrouter import OpenRouter
    eval_model = OpenRouter(id=cfg.model, api_key=cfg.openrouter_api_key)
else:
    from agno.models.ollama import Ollama
    eval_model = Ollama(id=cfg.model, host=cfg.ollama_host)

answer_agent = Agent(
    model=eval_model,
    instructions="Answer concisely based only on the context provided.",
)
grade_agent = Agent(
    model=eval_model,
    instructions="Return exactly CORRECT or WRONG as your final answer.",
)
print("Agents ready.")

## 7. Step 2a: Run one question (recall → generate → grade)

In [ ]:
ANSWER_PROMPT = """You are a helpful expert answering questions based on the provided context.
Context:
{context}

Question: {question}
Answer:"""

GRADE_PROMPT = """Label the answer as CORRECT or WRONG.
Question: {question}
Gold: {gold}
Generated: {response}
Return CORRECT or WRONG."""

user_id = "locomo_0"
qa = data[0]["qa"][0]
question = qa["question"]
gold = qa["answer"]

print("Question:", question)
print("Gold:", gold)
print()

print("Recalling...")
context = memory.recall(user_id, query=question)
print(f"Context length: {len(context)} chars")
print()

print("Generating answer...")
prompt = ANSWER_PROMPT.format(context=context, question=question)
answer_result = answer_agent.run(prompt)
answer = (answer_result.content or "").strip()
print("Answer:", answer)
print()

print("Grading...")
grade_prompt = GRADE_PROMPT.format(question=question, gold=gold, response=answer)
grade_result = grade_agent.run(grade_prompt)
text = (grade_result.content or "").strip().upper()
grade = "CORRECT" in text and "WRONG" not in text and "INCORRECT" not in text
print(f"Grade: {"CORRECT" if grade else "WRONG"}")

## 8. Step 2b: Run all questions

In [ ]:
CATEGORY_NAMES = {1: "single_hop", 2: "temporal", 3: "multi_hop", 4: "open_domain", 5: "abstention"}

def generate_answer(agent, context, question):
    p = ANSWER_PROMPT.format(context=context, question=question)
    r = agent.run(p)
    return (r.content or "").strip()

def grade_answer(agent, question, gold, response):
    p = GRADE_PROMPT.format(question=question, gold=str(gold), response=response)
    r = agent.run(p)
    t = (r.content or "").strip().upper()
    return "CORRECT" in t and "WRONG" not in t and "INCORRECT" not in t

results = defaultdict(list)
scores_by_cat = defaultdict(list)
step_start = time.monotonic()
done = 0

for group_idx in range(num_users):
    user_id = f"locomo_{group_idx}"
    qa_filtered = [q for q in data[group_idx].get("qa", []) if q.get("category") != 5]
    for qa in qa_filtered:
        question = qa.get("question", "")
        gold = qa.get("answer")
        if gold is None:
            continue
        context = memory.recall(user_id, query=question)
        answer = generate_answer(answer_agent, context, question)
        grade = grade_answer(grade_agent, question, gold, answer)
        done += 1
        cat_name = CATEGORY_NAMES.get(qa.get("category", 0), "unknown")
        scores_by_cat[cat_name].append(grade)
        results[user_id].append({"question": question, "answer": answer, "golden_answer": gold, "grade": grade, "category": cat_name})
        if done % 50 == 0 or done == total_qa:
            print(f"Progress: {done}/{total_qa} | {time.monotonic() - step_start:.0f}s")

print(f"\nQA done in {time.monotonic() - step_start:.1f}s")

## 9. Step 3: Aggregate and save results

In [ ]:
total = sum(len(v) for v in results.values())
correct = sum(1 for items in results.values() for it in items if it["grade"])
overall = correct / total if total else 0

by_cat = {
    name: sum(s) / len(s) if s else 0
    for name, s in scores_by_cat.items()
    if name != "abstention"
}

print("=== LoCoMo Eval Results ===")
print(f"Overall: {overall:.2%} ({correct}/{total})")
for name, acc in by_cat.items():
    print(f"  {name}: {acc:.2%}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_file = OUTPUT_DIR / "locomo_results.json"
with open(out_file, "w") as f:
    json.dump({
        "overall_accuracy": overall,
        "total_questions": total,
        "correct": correct,
        "by_category": by_cat,
        "results": dict(results),
    }, f, indent=2)
print(f"\nResults saved to {out_file}")